# CapCap GPU Server (Colab)
Sử dụng sổ tay này để chạy CapCap Remote API Server trên Google Colab nhằm tận dụng GPU miễn phí cho các tác vụ như Bóc băng (Whisper), Dịch thuật, OCR.

**Hướng dẫn:**
1. Vào menu **Runtime** > **Change runtime type** > Chọn **T4 GPU**.
2. Bấm nút **Play** ở ô bên dưới để chạy mã.
3. Đợi vài phút, hệ thống sẽ cấp cho bạn 1 URL (đuôi `.trycloudflare.com`) và 1 Token.
4. Mở file `.env` trên máy tính cục bộ của bạn, dán URL và Token đó vào `CAPCAP_REMOTE_API_URL` và `CAPCAP_REMOTE_API_TOKEN`.

In [ ]:
import os
import subprocess
import time
import secrets
import IPython.display as display

# 1. Tải CapCap và cài đặt môi trường
print("Đang tải mã nguồn CapCap và cài đặt thư viện (có thể mất vài phút)...")
!git clone https://github.com/khoinguyen59/KOVA-STUDIO.git
%cd KOVA-STUDIO
!pip install -r requirements-local.txt

# Cài đặt cloudflared
!curl -L --output cloudflared.deb https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared.deb

# 2. Tạo Token bảo mật và thiết lập môi trường
TOKEN = secrets.token_urlsafe(32)
os.environ["CAPCAP_REMOTE_API_TOKEN"] = TOKEN
os.environ["CAPCAP_REMOTE_API_PORT"] = "8765"
os.environ["CAPCAP_REMOTE_API_HOST"] = "127.0.0.1"
os.environ["CAPCAP_QUIET"] = "false"
os.environ["CAPCAP_RUNTIME_PROFILE"] = "remote"

# 3. Khởi chạy CapCap Remote API Server dưới nền
print("Đang khởi động CapCap Remote API Server...")
server_process = subprocess.Popen(
    ["python", "app/remote_api_server.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)
time.sleep(5)

# 4. Khởi chạy Cloudflare Tunnel
print("Đang thiết lập Cloudflare Tunnel...")
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8765", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# 5. Lấy URL Public từ Cloudflare
public_url = ""
while True:
    line = tunnel.stdout.readline()
    if not line:
        break
    if "https://" in line and ".trycloudflare.com" in line:
        import re
        match = re.search(r"https://[^\s\"']+\.trycloudflare\.com", line)
        if match:
            public_url = match.group(0)
            break

display.clear_output()
print("\n" + "="*70)
print("✅ MÁY CHỦ COLAB CAPCAP ĐÃ SẴN SÀNG ✅")
print("Hãy copy 2 dòng sau và dán vào file .env trên máy tính của bạn:")
print("="*70)
print(f"CAPCAP_REMOTE_API_URL={public_url}")
print(f"CAPCAP_REMOTE_API_TOKEN={TOKEN}")
print("="*70)
print("\nBạn có thể để tab này chạy nền. Đừng đóng trình duyệt nhé!\n")

# Hiển thị log của server liên tục
try:
    for line in iter(server_process.stdout.readline, ""):
        print(line, end="")
except KeyboardInterrupt:
    print("\nĐã dừng server.")
